# 찾아 쓰는 Materials Project API 예제
AI for Materials Science — Hands-on session 1 부록

수업 진도는 `2026_2_Hands_on_session1_inclass.ipynb`의 A–D절까지입니다.
이 노트북은 **필요할 때 찾아 쓰는 예제 모음**이라 위에서부터 다 실행할 필요가 없습니다.
원하는 절만 골라 실행하세요. 다만 **0절 준비는 항상 먼저 실행해야 합니다.**

## 어떤 예제가 있나

### 1 · 어떤 경로가 있는지 훑어보기
`MPRester`가 제공하는 도우미 함수와 검색 경로 목록을 확인합니다.

### 2 · 구조 내려받아 확인하고 CIF로 저장하기
### 3 · 계산 출처 따라가기 (thermo · provenance · task)
### 4 · 원하는 물성 데이터가 있는 후보부터 찾기 (탄성·유전·압전·자성)
### 5 · 전자 band structure와 DOS
### 6 · phonon band structure와 DOS
### 7 · XAS와 전하 밀도
### 8 · 전극 · 표면 · 자연어 구조 설명
### 9 · 물속에서의 안정성: Pourbaix diagram
### 10 · 다른 원소계와 4원계 상평형도로 확장하기

---

### 실행하기 전에 알아둘 것
API 함수는 버전에 따라 이름과 인자가 바뀔 수 있고, **모든 material에 모든 물성이 있는 것도 아닙니다.**
그래서 각 예제는 `available_fields`와 `inspect.signature`로 먼저 확인하는 흐름을 함께 담았습니다.

조회 결과가 비어 있는 것도 하나의 결과입니다. "데이터가 없다"는 사실도 그대로 기록하세요.

자세한 사용법은
[MPRester 문서](https://materialsproject.github.io/api/_autosummary/mp_api.client.mprester.MPRester.html)와
[공식 API 예제](https://docs.materialsproject.org/downloading-data/using-the-api/examples)에 있습니다.

## 0. 준비

### 0-1. 라이브러리 설치
4원계 상평형도를 대화형 그림으로 저장할 때 `plotly`가 필요해 함께 설치합니다.

In [ ]:
!pip install -q pymatgen mp_api plotly

### 0-2. 라이브러리 불러오기
아래 예제 전체가 쓰는 도구를 한 번에 가져옵니다.
전자구조·phonon·Pourbaix 도구는 파일이 무거워서, 해당 절 안에서 필요할 때만 따로 `import`합니다.

In [ ]:
import os
import inspect
from pathlib import Path
from getpass import getpass

import pandas as pd
import matplotlib.pyplot as plt

from pymatgen.core import Element
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer
from pymatgen.analysis.phase_diagram import PhaseDiagram, PDPlotter
from mp_api.client import MPRester

### 0-3. API 키와 저장 폴더
이 노트북은 모든 예제가 Materials Project에 직접 조회하므로 키가 반드시 필요합니다.
키는 [MP 계정 페이지](https://next-gen.materialsproject.org/api)에서 발급받을 수 있습니다.

In [ ]:
API_KEY = os.getenv("MP_API_KEY", "").strip()
if not API_KEY:
    API_KEY = getpass("Materials Project API key: ").strip()

if not API_KEY:
    raise ValueError("MP API 키가 필요합니다. 키를 발급받은 뒤 이 셀을 다시 실행해주세요.")

OUTPUT = Path("outputs/02_api_examples")
OUTPUT.mkdir(parents=True, exist_ok=True)

with MPRester(API_KEY) as mpr:
    print("MP DB version:", mpr.db_version)

### 0-4. 큰 자료를 받을지 정하기
전자구조, phonon, 전하밀도, Pourbaix, 4원계 상평형도는 파일이 크고 시간이 걸립니다.
5·6·7·9·10절의 무거운 셀은 아래 설정이 `True`일 때만 실행됩니다.

처음에는 `False`로 두고 가벼운 예제부터 보다가, 필요할 때 `True`로 바꾸고 이 셀부터 다시 실행하세요.

In [ ]:
RUN_LARGE_DOWNLOADS = False
print("큰 자료 다운로드:", "실행함" if RUN_LARGE_DOWNLOADS else "건너뜀")

## 1. 어떤 경로가 있는지 훑어보기

MP API는 목적에 따라 들어가는 문이 다릅니다. 대략 이렇게 나뉩니다.

- **조성·구조·대칭·물성 조건 검색** — `materials.summary`, `get_structure_by_material_id`
- **계산 에너지·보정·convex hull** — `materials.thermo`, `get_entries`, `get_entries_in_chemsys`
- **개별 계산·출처·논문** — `materials.tasks`, `materials.provenance`, `get_material_id_references`
- **탄성·유전·압전·자성** — `materials.elasticity`, `dielectric`, `piezoelectric`, `magnetism`
- **전자 band structure / DOS** — `get_bandstructure_by_material_id`, `get_dos_by_material_id`
- **phonon band structure / DOS** — `get_phonon_bandstructure_by_material_id`, `get_phonon_dos_by_material_id`
- **XAS·전하 밀도** — `materials.xas`, `get_charge_density_from_material_id`
- **전지** — `materials.insertion_electrodes`, `conversion_electrodes`
- **표면·형태** — `materials.surface_properties`, `get_wulff_shape`
- **수용액 안정성** — `get_pourbaix_entries` → `PourbaixDiagram`
- **자연어 구조 설명** — `materials.robocrys`

### 1-1. 설치된 버전에서 실제로 쓸 수 있는 것 확인하기
문서보다 **지금 설치된 버전**이 우선입니다. `dir`로 이름을 직접 확인하는 습관을 들이세요.

In [ ]:
## get_으로 시작하는 이름이 곧바로 쓸 수 있는 도우미 함수입니다.
print("MPRester의 get_* 도우미:")
print([name for name in dir(MPRester) if name.startswith("get_")])

In [ ]:
with MPRester(API_KEY) as mpr:
    ## 밑줄로 시작하는 내부용 이름은 빼고 정렬합니다.
    route_names = sorted(name for name in dir(mpr.materials) if not name.startswith("_"))
    print("materials 아래의 검색 경로:")
    print(route_names)

In [ ]:
## 경로 하나를 골라 어떤 조건을 넣고 무엇을 받을 수 있는지 확인해 봅니다.
with MPRester(API_KEY) as mpr:
    print("elasticity.search 조건:", inspect.signature(mpr.materials.elasticity.search))
    print()
    print("elasticity 출력 항목:", mpr.materials.elasticity.available_fields)

## 2. 구조를 내려받아 확인하고 CIF로 저장하기

`get_structure_by_material_id`는 material ID 하나로 pymatgen `Structure`를 바로 돌려줍니다.
연습용으로 실리콘 `mp-149`를 받아 원자 수, 격자, 밀도, 공간군을 확인해 보겠습니다.

`symprec`은 대칭을 판정할 때 허용할 원자 위치 오차(Å)입니다.
**API에서 받은 구조가 이미 표준화되어 있어도 이 값에 따라 공간군 판정이 달라질 수 있습니다.**
그래서 공간군을 보고할 때는 쓴 `symprec`도 함께 적어야 합니다.

In [ ]:
with MPRester(API_KEY) as mpr:
    structure = mpr.get_structure_by_material_id("mp-149")

print(structure.composition.reduced_formula, "| 원자 자리 수:", len(structure))
print("격자 길이 (Å):", structure.lattice.abc)
print("밀도 (g/cm^3):", float(structure.density))

In [ ]:
## symprec을 바꿔 가며 판정이 달라지는지 확인해 보세요.
for symprec in [0.01, 0.1]:
    analyzer = SpacegroupAnalyzer(structure, symprec=symprec)
    print(f"symprec={symprec}: {analyzer.get_space_group_symbol()} "
          f"({analyzer.get_space_group_number()})")

In [ ]:
## 받은 구조를 CIF로 저장하면 VESTA 같은 프로그램에서도 열 수 있습니다.
structure.to(filename=str(OUTPUT / "mp-149_Si.cif"), fmt="cif")
print("저장:", OUTPUT / "mp-149_Si.cif")

## 3. 계산 출처 따라가기: thermo · provenance · task

**하나의 material에는 여러 계산이 연결되어 있습니다.**
summary에 보이는 값이 어느 계산에서 나온 것인지 따라가 보는 것이 이 절의 목적입니다.

`thermo_type`도 함께 보세요. `GGA_GGA+U_R2SCAN`은 여러 계산을 섞어 만든 열역학 자료이며,
**순수 r2SCAN의 다른 이름이 아닙니다.**

In [ ]:
with MPRester(API_KEY) as mpr:
    thermo_docs = mpr.materials.thermo.search(
        material_ids=["mp-149"],
        fields=["material_id", "thermo_type", "formation_energy_per_atom", "energy_above_hull"],
        all_fields=False, num_chunks=1, chunk_size=20)

## model_dump는 문서 객체를 딕셔너리로 바꿔 줍니다. 에너지 단위는 eV/atom입니다.
pd.DataFrame([doc.model_dump() for doc in thermo_docs])

In [ ]:
with MPRester(API_KEY) as mpr:
    ## calc_types는 이 material에 어떤 종류의 계산이 딸려 있는지 알려 줍니다.
    core_docs = mpr.materials.search(material_ids=["mp-149"],
                                     fields=["material_id", "calc_types"], all_fields=False)
    ## provenance에는 원 논문 정보가 들어 있습니다.
    provenance_docs = mpr.materials.provenance.search(material_ids=["mp-149"],
                                                      fields=["material_id", "references"],
                                                      all_fields=False)

print("calc_types:", [doc.calc_types for doc in core_docs])
print("provenance 문서 수:", len(provenance_docs))

### 3-1. summary의 `origins`로 실제 계산까지 따라가기
`origins`는 "이 물성값이 어느 task에서 왔는가"를 알려 줍니다.
여기서 task ID를 꺼내 개별 계산의 입력과 출력까지 확인할 수 있습니다.

In [ ]:
with MPRester(API_KEY) as mpr:
    origin_docs = mpr.materials.summary.search(material_ids=["mp-149"],
                                               fields=["material_id", "origins"],
                                               all_fields=False)
    ## 문서가 없을 수도 있으므로 빈 목록을 기본값으로 둡니다.
    origins = origin_docs[0].origins if origin_docs else []
    print("물성값의 출처:", origins)

    ## task_id가 있는 출처만 골라 앞의 두 개만 확인합니다.
    task_ids = [str(o.task_id) for o in origins if getattr(o, "task_id", None)][:2]
    task_docs = (mpr.materials.tasks.search(task_ids=task_ids,
                                            fields=["task_id", "input", "output"],
                                            all_fields=False) if task_ids else [])

print("조회한 task:", [str(doc.task_id) for doc in task_docs])

## 4. 원하는 물성 데이터가 있는 후보부터 찾기

물성 조건으로 검색하려면 먼저 **그 물성 데이터가 존재하는 재료**를 찾아야 합니다.
`has_props`가 그 역할을 합니다.

`has_props`(데이터가 있는가)와 물성 범위 조건(값이 얼마인가)은 다른 것입니다.
**값이 0인 경우와 데이터가 없는 경우도 구분해야 합니다.**

아래에서는 산소를 포함한 후보를 물성별로 세 개씩 찾은 뒤, 각 상세 endpoint에서 문서를 받아 옵니다.
후보 ID 수와 상세 문서 수가 항상 같지는 않다는 점을 확인하세요.

In [ ]:
## 물성 이름과 검색 경로 이름의 쌍입니다. 여기서는 네 물성에 같은 과정을 반복합니다.
PROPERTY_ROUTES = [("elasticity", "elasticity"), ("dielectric", "dielectric"),
                   ("piezoelectric", "piezoelectric"), ("magnetism", "magnetism")]

property_examples = {}
with MPRester(API_KEY) as mpr:
    for prop, route_name in PROPERTY_ROUTES:
        candidates = mpr.materials.summary.search(
            elements=["O"], has_props=[prop],
            fields=["material_id", "formula_pretty"], all_fields=False,
            num_chunks=1, chunk_size=3)
        ids = [str(d.material_id) for d in candidates]

        ## getattr로 문자열 이름에 해당하는 검색 경로를 가져옵니다.
        route = getattr(mpr.materials, route_name)
        detail = route.search(material_ids=ids, num_chunks=1, chunk_size=3) if ids else []
        property_examples[prop] = detail
        print(f"{prop:14s} 후보 {len(ids)}건 -> 상세 문서 {len(detail)}건  {ids}")

In [ ]:
## 각 경로가 어떤 항목을 돌려주는지 확인해 두면 fields를 좁힐 때 편합니다.
with MPRester(API_KEY) as mpr:
    for prop, route_name in PROPERTY_ROUTES:
        fields = getattr(mpr.materials, route_name).available_fields
        print(f"{prop:14s} ({len(fields)}개): {fields[:8]} ...")

## 5. 전자 band structure와 DOS

파일이 커서 내려받고 그리는 데 시간이 걸립니다. **0-4의 `RUN_LARGE_DOWNLOADS`를 `True`로 바꿔야 실행됩니다.**

두 그림은 같은 계산의 다른 표현입니다.

- **band structure** — 고대칭 k점을 잇는 경로를 따라 에너지를 그린 것
- **DOS** — 전체 k점 정보를 에너지 축으로 모아 셈한 것

계산 band gap을 실험값과 같은 값으로 해석하면 안 됩니다.
GGA 계열은 band gap을 실제보다 작게 예측하는 경향이 있습니다.

In [ ]:
if RUN_LARGE_DOWNLOADS:
    from pymatgen.electronic_structure.plotter import BSPlotter, DosPlotter

    with MPRester(API_KEY) as mpr:
        bs = mpr.get_bandstructure_by_material_id("mp-149")
        dos = mpr.get_dos_by_material_id("mp-149")

    ## None은 자료가 없다는 뜻입니다. 있을 때만 그립니다.
    if bs is not None:
        BSPlotter(bs).get_plot()
        plt.show()
        print("band gap:", bs.get_band_gap())

    if dos is not None:
        dp = DosPlotter()
        dp.add_dos("Si total DOS", dos)
        dp.get_plot()
        plt.show()
else:
    print("RUN_LARGE_DOWNLOADS=False 이므로 건너뜁니다.")

## 6. phonon band structure와 DOS

phonon 자료는 모든 material에 있지 않습니다.
그래서 **먼저 문서가 있는지 확인하고**, 있을 때만 내려받는 순서로 진행합니다.

그림에서 **음수로 표시된 진동수(imaginary mode)**를 눈여겨보세요.
동역학적으로 불안정하거나 계산이 충분히 수렴하지 않았다는 신호입니다.

In [ ]:
if RUN_LARGE_DOWNLOADS:
    from pymatgen.phonon.plotter import PhononBSPlotter, PhononDosPlotter

    with MPRester(API_KEY) as mpr:
        summary_ph = mpr.materials.summary.search(material_ids=["mp-149"],
                                                  fields=["material_id", "phonon_IDs"],
                                                  all_fields=False)
        ## phonon_IDs가 없거나 비어 있어도 빈 목록이 되도록 기본값을 둡니다.
        dfpt_ids = ((summary_ph[0].phonon_IDs or {}).get("dfpt", []) if summary_ph else [])

        if dfpt_ids:
            phonon_docs = mpr.materials.phonon.search(identifiers=dfpt_ids[:1],
                                                      fields=["identifier"], all_fields=False,
                                                      num_chunks=1, chunk_size=1)
            print("DFPT phonon identifiers:", [doc.identifier for doc in phonon_docs])
            ph_bs = mpr.get_phonon_bandstructure_by_material_id("mp-149")
            ph_dos = mpr.get_phonon_dos_by_material_id("mp-149")
        else:
            ph_bs, ph_dos = None, None
            print("이 material에는 phonon 문서가 없습니다. 다른 ID를 찾아보세요.")

    if ph_bs is not None:
        PhononBSPlotter(ph_bs).get_plot()
        plt.show()
    if ph_dos is not None:
        pp = PhononDosPlotter()
        pp.add_dos("Si phonon DOS", ph_dos)
        pp.get_plot()
        plt.show()
else:
    print("RUN_LARGE_DOWNLOADS=False 이므로 건너뜁니다.")

## 7. XAS와 전하 밀도

XAS(X선 흡수 분광)는 **흡수 원소**와 **absorption edge**를 함께 지정해야 합니다.
조성만으로는 어느 원소의 어느 흡수단인지 정해지지 않기 때문입니다.

아래는 TiO₂의 Ti K-edge 스펙트럼입니다. 축의 단위를 확인하세요.
x는 광자 에너지(eV), y는 상대 세기(임의 단위)입니다.

In [ ]:
with MPRester(API_KEY) as mpr:
    ## Element("Ti")로 원소 객체를 만들어 흡수 원소를 지정합니다.
    spectra = mpr.materials.xas.search(formula="TiO2", absorbing_element=Element("Ti"),
                                       edge="K", num_chunks=1, chunk_size=1)

print("Ti K-edge 문서 수:", len(spectra))

if spectra:
    spectrum = spectra[0].spectrum
    fig, ax = plt.subplots(figsize=(6, 3))
    ax.plot(spectrum.x, spectrum.y)
    ax.set(xlabel="Photon energy (eV)", ylabel="Intensity (a.u.)", title="Ti K-edge XAS")
    fig.tight_layout()
    plt.show()

전하 밀도는 3차원 격자 위의 값이라 파일이 매우 큽니다. 큰 자료 설정이 켜져 있을 때만 받습니다.
저장 형식인 CHGCAR는 VASP가 쓰는 전하밀도 파일 형식입니다.

In [ ]:
if RUN_LARGE_DOWNLOADS:
    with MPRester(API_KEY) as mpr:
        charge = mpr.get_charge_density_from_material_id("mp-149")

    if charge is not None:
        charge.write_file(str(OUTPUT / "mp-149_CHGCAR"))
        print("저장:", OUTPUT / "mp-149_CHGCAR")
    else:
        print("이 material에는 전하밀도 자료가 없습니다.")
else:
    print("RUN_LARGE_DOWNLOADS=False 이므로 건너뜁니다.")

## 8. 전극 · 표면 · 자연어 구조 설명

**endpoint마다 쓰는 ID 체계가 다를 수 있습니다.**
특히 전극 결과에는 material ID가 아니라 battery ID와 충전·방전 상태 정보가 들어 있으니,
출력된 열 이름을 먼저 확인하세요.

In [ ]:
with MPRester(API_KEY) as mpr:
    ## 작동 이온이 Li이고 Fe·P·O를 포함하며 평균 전압이 2.5-4.5 V인 전극 후보입니다.
    batteries = mpr.materials.insertion_electrodes.search(
        elements=["Fe", "P", "O"], working_ion=Element("Li"),
        average_voltage=(2.5, 4.5), num_chunks=1, chunk_size=3)

print("Li 삽입형 전극 문서 수:", len(batteries))
pd.DataFrame([doc.model_dump() for doc in batteries]).head() if batteries else "결과 없음"

In [ ]:
with MPRester(API_KEY) as mpr:
    ## 표면 물성은 material ID로 찾습니다.
    surfaces = mpr.materials.surface_properties.search(material_ids=["mp-149"],
                                                       num_chunks=1, chunk_size=1)
    ## robocrys는 구조를 사람이 읽는 문장으로 설명해 주는 자료입니다.
    descriptions = mpr.materials.robocrys.search(keywords=["perovskite"],
                                                 num_chunks=1, chunk_size=1)

print("Si 표면 문서 수:", len(surfaces))
print("구조 설명 검색 결과:", len(descriptions))

## 9. 물속에서의 안정성: Pourbaix diagram

지금까지 본 상평형도는 고체끼리의 경쟁만 다뤘습니다.
Pourbaix diagram은 여기에 **pH와 전위**를 축으로 더해, 수용액 환경에서 무엇이 안정한지 봅니다.

용존 이온을 함께 다루기 때문에 **이온 농도를 반드시 명시해야 합니다.**
농도를 바꾸면 안정 영역의 경계가 움직입니다.

아래 예제는 Fe 농도를 10⁻⁶ mol/L로 두었습니다. 부식 연구에서 흔히 쓰는 기준 농도입니다.

In [ ]:
if RUN_LARGE_DOWNLOADS:
    from pymatgen.analysis.pourbaix_diagram import PourbaixDiagram, PourbaixPlotter

    with MPRester(API_KEY) as mpr:
        aqueous_entries = mpr.get_pourbaix_entries(["Fe"])
    print("수용액계 entry 수:", len(aqueous_entries))

    ## conc_dict의 단위는 mol/L입니다.
    aqueous_pd = PourbaixDiagram(aqueous_entries, conc_dict={"Fe": 1e-6})

    ## limits의 첫 쌍은 pH 범위, 둘째 쌍은 전위 범위(V)입니다.
    PourbaixPlotter(aqueous_pd).get_pourbaix_plot(limits=[[0, 14], [-2, 2]])
    plt.show()
else:
    print("RUN_LARGE_DOWNLOADS=False 이므로 건너뜁니다.")

## 10. 다른 원소계와 4원계 상평형도로 확장하기

수업 C절에서는 배포된 계산 자료로 Li–Fe–O를 그렸습니다.
현재 MP를 직접 조회하면 **원하는 원소계를 아무거나** 그릴 수 있습니다.

### 10-1. Li–Co–O 삼원계
`get_entries_in_chemsys`는 지정한 원소의 **모든 부분계**를 함께 가져옵니다.
C절에서 설명했듯 이 부분이 빠지면 convex hull을 만들 수 없습니다.

In [ ]:
PD_ELEMENTS = ["Li", "Co", "O"]
THERMO_TYPE = "GGA_GGA+U"

with MPRester(API_KEY) as mpr:
    ## thermo_types로 어떤 계산 자료를 쓸지 명시합니다. 섞어 쓰면 에너지 기준이 달라집니다.
    entries = mpr.get_entries_in_chemsys(PD_ELEMENTS, compatible_only=True,
                                         additional_criteria={"thermo_types": [THERMO_TYPE]})
    db_version = mpr.db_version

print(f"{'-'.join(PD_ELEMENTS)} | {THERMO_TYPE} | DB {db_version}")
print("entry 수:", len(entries))

In [ ]:
phase_diagram = PhaseDiagram(entries)
print("안정 entry 수:", len(phase_diagram.stable_entries))

ax = PDPlotter(phase_diagram, backend="matplotlib",
               show_unstable=False).get_plot(label_stable=True)
ax.figure.set_size_inches(8, 7)
ax.figure.savefig(OUTPUT / f"{'-'.join(PD_ELEMENTS)}_phase_diagram.png",
                  dpi=180, bbox_inches="tight")
plt.show()

### 10-2. 4원계 Li–Fe–P–O
원소가 넷이면 삼각형 하나로 표현할 수 없습니다. 사면체가 되기 때문에 3차원 그림이 필요합니다.
`backend="plotly"`로 만들면 브라우저에서 돌려 가며 볼 수 있는 HTML이 나옵니다.

**이 조회는 entry를 많이 받습니다.** 시간이 걸리므로 큰 자료 설정을 켰을 때만 실행됩니다.

받는 양을 줄이겠다고 `is_stable=True`로 거르거나 검색 결과의 일부 페이지만 쓰면 안 됩니다.
경쟁상이 빠진 hull은 잘못된 안정성을 알려 줍니다.

In [ ]:
if RUN_LARGE_DOWNLOADS:
    with MPRester(API_KEY) as mpr:
        lfp_entries = mpr.get_entries_in_chemsys(
            ["Li", "Fe", "P", "O"],
            additional_criteria={"thermo_types": ["GGA_GGA+U"]})
    print("entry 수:", len(lfp_entries))

    lfp_pd = PhaseDiagram(lfp_entries)
    lfp_plot = PDPlotter(lfp_pd, backend="plotly", show_unstable=False).get_plot()

    ## HTML로 저장하면 나중에 브라우저에서 다시 열 수 있습니다.
    html_path = OUTPUT / "Li-Fe-P-O_4component_phase_diagram.html"
    lfp_plot.write_html(str(html_path))
    print("저장:", html_path)
    lfp_plot.show()
else:
    print("RUN_LARGE_DOWNLOADS=False 이므로 건너뜁니다.")